### 1. load datasets

In [2]:
import pandas as pd

import pandas as pd
df_en = pd.read_csv('../../datasets/translation/pure_english.csv')
print(df_en.shape)
df_en.head()

(64, 5)


,scheme_id,site,scheme_name,description,scheme_link
0,1,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,A dose of happiness,d.As per the government decision dated 04.10.2...,https://mahafood.gov.in/scheme/%e0%a4%86%e0%a4...
1,2,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,APL Farmers,Mahe per beneficiary per month instead of food...,https://mahafood.gov.in/scheme/%e0%a4%8f%e0%a4...
2,3,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,Shiv Bhojan,To provide food at discounted rates to the poo...,https://mahafood.gov.in/scheme/%e0%a4%b6%e0%a4...
3,4,https://maharashtra.gov.in/Site/1604/scheme,Agriculture Scheme,Many schemes are available by the state govern...,https://www.manage.gov.in/fpoacademy/SGSchemes...
4,5,https://maharashtra.gov.in/Site/1604/scheme,Agricultural Mortgage Loan Scheme,Due to the financial need of the farmer and la...,https://www.msamb.com/Schemes/PledgeFinance


### 2. connect elasticsearch

In [3]:
from elasticsearch import Elasticsearch
es = Elasticsearch("http://localhost:9200",
                   basic_auth = ('elasticsearch', 'e3TKzHmKRFWBP4gY--cjeQ'),
                   request_timeout=60,
                   )
es.ping() 
print(es.info())

{'name': 'LAPTOP-ANN0J427', 'cluster_name': 'elasticsearch', 'cluster_uuid': '6AxnonBCTU-8fGsKa_EYMQ', 'version': {'number': '9.1.3', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': '0c781091a2f57de895a73a1391ff8426c0153c8d', 'build_date': '2025-08-24T22:05:04.526302670Z', 'build_snapshot': False, 'lucene_version': '10.2.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


  ### 3. load models

In [15]:
import sys, os
sys.path.append(os.path.abspath("../loadModels"))

In [4]:
from loadModels import load_mahaSBERT

mahaSBERT_model = load_mahaSBERT()
print("mahaSBERT model loaded successfully")


mahaSBERT model loaded successfully


In [5]:
from loadModels import load_indicSBERT

indicSBERT_model = load_indicSBERT()

print("indicSBERT model loaded successfully")

indicSBERT model loaded successfully


### 4. generate embeddings


In [6]:
df_en["mahasbert_des_vector"] = df_en["description"].apply(lambda x: mahaSBERT_model.encode(x))

In [7]:
df_en["indicsbert_des_vector"] = df_en["description"].apply(lambda x: indicSBERT_model.encode(x))

### 5. generate mappings

In [8]:
from indexMappings import indexMappings

es.indices.create(index = "english" , mappings = indexMappings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'english'})

### 6. Generate records

In [10]:
english_record_list = df_en.to_dict("records")

In [11]:
english_record_list[0]["mahasbert_des_vector"]

array([-2.16954090e-02, -6.71289861e-04,  2.18075607e-02,  1.68130901e-02,
        1.40578272e-02, -9.87081975e-03, -4.37575020e-03,  6.91618444e-03,
        1.50281778e-02, -1.19368988e-03, -4.42514848e-03,  1.22459624e-02,
        1.21826800e-02, -3.25396005e-03,  1.96121400e-03, -1.49512766e-02,
        9.94591392e-04, -7.05431495e-03,  4.26555751e-03, -1.08227665e-02,
        5.37820579e-03,  1.42841449e-03, -1.11833122e-02,  8.75128992e-03,
        1.15058972e-02,  1.35949533e-02,  2.41236798e-02, -2.69348570e-03,
        9.15739499e-03,  1.59478858e-02, -1.63425449e-02, -1.25961285e-02,
        6.77140662e-03, -2.00131116e-03, -2.92951777e-03, -1.10876039e-02,
        4.13741014e-04, -1.29094021e-02,  1.77602060e-02, -1.08561553e-02,
        7.38509605e-03, -5.12687396e-03, -2.09938027e-02, -1.86493546e-02,
       -6.88396976e-04, -2.86698304e-02,  1.82559770e-02,  1.36670172e-02,
       -5.76782646e-03, -2.37056264e-03, -9.40783229e-03, -1.55136818e-02,
       -1.09151835e-02,  

In [12]:
english_record_list[0]["indicsbert_des_vector"]

array([-2.19422020e-02, -6.15258981e-03,  1.53254112e-02, -7.13874726e-03,
        1.07729919e-02,  2.42509297e-03,  7.43923010e-05, -7.52063189e-03,
        5.22302650e-03, -9.14558861e-03,  9.52869933e-03,  8.25064909e-03,
        8.75031669e-03,  1.10405087e-02, -4.83039883e-04, -1.30387908e-02,
       -1.04949810e-02, -1.20956050e-02, -1.18583580e-02, -1.32361827e-02,
        3.23529297e-04,  6.40916824e-03, -2.38987599e-02,  4.81201988e-03,
        6.62191631e-03,  3.98336444e-03,  2.16898378e-02, -1.35626504e-02,
        1.86431105e-03,  4.92710108e-03, -1.27891609e-02, -1.15672946e-02,
        1.82303705e-03, -2.10639485e-03,  1.00974338e-02, -1.27202198e-02,
       -9.50566214e-03,  6.17636088e-03,  3.28562711e-03, -6.91738632e-03,
        9.83406603e-03, -7.54402764e-03, -2.42563542e-02, -2.15629786e-02,
        2.57191155e-03, -1.69616491e-02,  2.09940393e-02,  1.18344845e-02,
        4.08730283e-03,  3.01981205e-03, -1.44387670e-02, -7.21065095e-03,
        1.24795118e-03, -

In [13]:
for record in english_record_list:
    try:
        es.index(index="english", document=record, id=record['scheme_id'])
    except Exception as e:
        print("error", e)

In [14]:
es.count(index="english")

ObjectApiResponse({'count': 64, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})